# SautiCivic Bridge — OpenAI Whisper large-v3 Benchmark (Google Colab)

This notebook runs the **OpenAI Whisper large-v3** open-weights model on Google Colab GPU.

### Why Colab GPU for Whisper:
- Whisper `large-v3` is a 1.5-billion parameter model (~3 GB weights) requiring GPU acceleration for fast inference without exhausting local laptop resources.
- In contrast, cloud API-based models (**Sahara v2.5**, **Deepgram Nova-3**, **Gemini 3.5 Transcribe**) can run directly on your local machine via their respective runners (`run_sahara.py`, `run_deepgram.py`, `run_gemini.py`).

### Colab Setup:
1. Go to **Runtime** > **Change runtime type** in the top menu.
2. Select **T4 GPU** (or A100/V100 if available) as the Hardware Accelerator.
3. Run the cells step-by-step.

## 1. Verify GPU Availability

In [ ]:
!nvidia-smi

## 2. Clone Repository & Verify 30 Audio Clips

In [ ]:
import os
from pathlib import Path

# 1. Clone repository in Colab if not already cloned
if not os.path.exists("Sauticivic") and not os.path.exists("bench"):
    !git clone https://github.com/Spyder0000/Sauticivic.git

# 2. Enter repository directory
if os.path.exists("Sauticivic"):
    %cd Sauticivic

# 3. Verify all 30 audio clips exist
corpus_dir = Path("bench/corpus/tier_a_recorded/audio")
if corpus_dir.is_dir():
    clips = sorted([f.name for f in corpus_dir.glob("*.wav")])
    print(f"✓ Found {len(clips)} audio clips in {corpus_dir}:")
    for clip in clips:
        print(f"  - {clip}")
else:
    print(f"⚠ Corpus directory not found at {corpus_dir}.")


## 3. Install Dependencies
Install `openai-whisper` and `ffmpeg`.

In [ ]:
!apt-get update -qq && apt-get install -y -qq ffmpeg
!pip install -q openai-whisper

## 4. Run Whisper large-v3 Across All 30 Clips
Transcribes all 30 Tier A recorded audio clips using GPU-accelerated Whisper `large-v3` and writes JSON results to `bench/results/transcripts/whisper/`.

In [ ]:
!python3 bench/models/run_whisper.py \
    --corpus bench/corpus/tier_a_recorded/audio \
    --output-dir bench/results/transcripts \
    --model large-v3

## 5. Package & Download Whisper Transcripts
Packages the resulting `whisper/` transcripts into a zip archive and downloads it to your machine.

In [ ]:
!zip -r whisper_transcripts.zip bench/results/transcripts/whisper/

from google.colab import files
files.download("whisper_transcripts.zip")

### Next Steps on Local Machine:
1. Extract `whisper_transcripts.zip` so the JSON files are placed in `bench/results/transcripts/whisper/`.
2. Run the complete benchmark orchestrator:
   ```bash
   PYTHONPATH=backend python3 -m bench.metrics.run_full_benchmark
   ```